In [1]:
import pandas as pd
from IPython.display import display

from doc_verify.config import SAD_SYN_DIR
from doc_verify.text_length import (
    CLIP_TEXT_TOKENS,
    EVIDENCE_FIELDS,
    evidence_end_token,
    load_tokenizer,
    token_count,
)
from doc_verify.tobacco import TOBACCO_OUT_DIR
from sad_syn.dataset import plan_documents
from sad_syn.ocr import expected_ocr_fields

tobacco = pd.read_csv(TOBACCO_OUT_DIR / "manifest.csv")
sad_syn = pd.read_csv(SAD_SYN_DIR / "manifest.csv")
tokenizer = load_tokenizer()

In [2]:
display(tobacco.groupby(['doc_type', 'split']).size().unstack()[['train', 'val', 'test']])

tobacco.groupby('doc_type').agg(
    pages=('split', 'size'),
    no_text=('mean_confidence', lambda c: c.isna().sum()),
    median_confidence=('mean_confidence', 'median'),
).round(1)

split,train,val,test
doc_type,,,
advertisement,161,34,35
email,419,90,90
form,302,65,64
letter,397,85,85
memo,434,93,93
news,132,28,28
note,141,30,30
report,186,40,39
resume,84,18,18


,pages,no_text,median_confidence
doc_type,,,
advertisement,230,6,64.9
email,599,0,90.6
form,431,2,73.9
letter,567,0,90.3
memo,620,0,89.7
news,188,1,81.9
note,201,6,46.6
report,265,0,91.8
resume,120,0,91.7


In [3]:
def read_text(root, path):
    return (root / path).read_text(encoding="utf-8")


tobacco['tokens'] = [token_count(read_text(TOBACCO_OUT_DIR, p), tokenizer) for p in tobacco.text_path]
sad_syn['tokens'] = [token_count(read_text(SAD_SYN_DIR, p), tokenizer) for p in sad_syn.text_path]

lengths = pd.concat([tobacco, sad_syn]).groupby(['source', 'doc_type'])['tokens']
pd.DataFrame({
    'median_tokens': lengths.median(),
    'q90_tokens': lengths.quantile(0.9),
    'fits_in_75': lengths.apply(lambda t: (t <= CLIP_TEXT_TOKENS).mean()),
    'windows_needed_median': lengths.median() / CLIP_TEXT_TOKENS,
}).round(2)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (99 > 77). Running this sequence through the model will result in indexing errors


median_tokens  q90_tokens  fits_in_75  \
source      doc_type                                                
sad_syn     bank_statement          312.0       353.0        0.00   
            offer_letter            238.0       252.0        0.00   
            transcript              256.0       294.0        0.00   
tobacco3482 advertisement            90.0       399.2        0.46   
            email                   150.0       437.0        0.13   
            form                    268.0       504.0        0.03   
            letter                  335.0       500.8        0.00   
            memo                    314.0       522.1        0.00   
            news                    949.5      1616.2        0.03   
            note                     52.0       206.0        0.61   
            report                  415.0       649.6        0.00   
            resume                  645.5      1127.9        0.00   
            scientific              528.0      1511.0        0.01   

                            windows_needed_median  
source      doc_type                               
sad_syn     bank_statement                   4.16  
            offer_letter                     3.17  
            transcript                       3.41  
tobacco3482 advertisement                    1.20  
            email                            2.00  
            form                             3.57  
            letter                           4.47  
            memo                             4.19  
            news                            12.66  
            note                             0.69  
            report                           5.53  
            resume                           8.61  
            scientific                       7.04

In [4]:
jobs = {str(job['image_path']): job for job in plan_documents(1500)}

rows = []
for r in sad_syn[sad_syn.label == 'altered'].itertuples():
    job = jobs[r.image_path]
    doc, template = job['doc'], job['template']
    fields = expected_ocr_fields(doc, template)
    values = [fields[f] for f in EVIDENCE_FIELDS[r.doc_type][r.inconsistency_type]]
    end = evidence_end_token(read_text(SAD_SYN_DIR, r.text_path), values, tokenizer)
    rows.append({'doc_type': r.doc_type, 'kind': r.inconsistency_type, 'split': r.split,
                 'page_tokens': r.tokens, 'evidence_end': end})
evidence = pd.DataFrame(rows)
evidence['readable'] = evidence.evidence_end.notna()
evidence['within_75'] = evidence.evidence_end <= CLIP_TEXT_TOKENS

evidence.groupby(['doc_type', 'kind']).agg(
    fakes=('readable', 'size'),
    evidence_readable=('readable', 'mean'),
    median_end_token=('evidence_end', 'median'),
    q90_end_token=('evidence_end', lambda e: e.quantile(0.9)),
    within_first_75=('within_75', 'mean'),
).round(2)

fakes  evidence_readable  \
doc_type       kind                                             
bank_statement altered_amount          120               0.71   
               altered_date            140               0.85   
               institution_mismatch    120               0.87   
               total_mismatch          120               0.52   
offer_letter   altered_amount          120               0.90   
               altered_date            140               0.90   
               institution_mismatch    120               0.88   
               total_mismatch          120               0.85   
transcript     altered_amount          120               0.92   
               altered_date            140               0.90   
               institution_mismatch    120               0.90   
               total_mismatch          120               0.91   

                                     median_end_token  q90_end_token  \
doc_type       kind                                                    
bank_statement altered_amount                   300.0          340.2   
               altered_date                      92.0          101.2   
               institution_mismatch             288.5          321.0   
               total_mismatch                   266.0          314.5   
offer_letter   altered_amount                   213.5          223.6   
               altered_date                     176.0          202.0   
               institution_mismatch              89.5          135.0   
               total_mismatch                   197.0          214.0   
transcript     altered_amount                   244.0          286.0   
               altered_date                      76.0           80.5   
               institution_mismatch             244.0          274.0   
               total_mismatch                    89.0          237.0   

                                     within_first_75  
doc_type       kind                                   
bank_statement altered_amount                   0.00  
               altered_date                     0.00  
               institution_mismatch             0.00  
               total_mismatch                   0.00  
offer_letter   altered_amount                   0.00  
               altered_date                     0.00  
               institution_mismatch             0.00  
               total_mismatch                   0.00  
transcript     altered_amount                   0.00  
               altered_date                     0.43  
               institution_mismatch             0.00  
               total_mismatch                   0.02

In [5]:
readable = evidence[evidence.readable]
print(f"fakes whose evidence OCR kept: {evidence.readable.mean():.1%}")
print(f"of those, evidence complete within the first 75 tokens: {readable.within_75.mean():.1%}")
print(f"median page length: {sad_syn.tokens.median():.0f} tokens; median evidence end: {readable.evidence_end.median():.0f} tokens")

fakes whose evidence OCR kept: 84.4%
of those, evidence complete within the first 75 tokens: 5.0%
median page length: 258 tokens; median evidence end: 198 tokens
